# XLS-R 300M fine-tuning on Tarifit Corpus V1.2

This notebook runs a fresh `facebook/wav2vec2-xls-r-300m` CTC model on the frozen V1.2 train/validation partitions.

**Experiment policy**

- The V1.2 train/validation metadata are reused exactly as frozen for the other controlled experiments.
- The final test partition is not loaded for model selection or training.
- The tokenizer is rebuilt specifically for the V1.2 final Tarifit character inventory.
- The default condition is **no augmentation**, so it can be compared with the main MMS, OmniASR, and Fadhma V1.2 adaptation references.
- A separate `specaugment` condition is available by changing one variable in Cell 2. It uses training-only time masking (`mask_time_prob=0.05`, `mask_time_length=5`) with feature masking disabled.
- The default run is trained for a planned maximum of eight epochs, with early stopping and checkpoint selection by validation CER.
        

In [ ]:
# Cell 1 — Install reproducible XLS-R V1.2 dependencies

!pip -q install "transformers==4.57.1" "datasets==4.4.1" "accelerate>=1.10,<2" "jiwer==4.0.0" "safetensors>=0.4.5" "soundfile>=0.12.1"


In [ ]:
# Cell 2 — Mount Google Drive and define XLS-R V1.2 paths

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm"
)

# Choose the comparison family before running the training cells.
# Use "noaug" for the main cross-model V1.2 comparison.
# Use "specaugment" only for the separate augmentation experiment.
AUGMENTATION_CONDITION = "noaug"

assert AUGMENTATION_CONDITION in {"noaug", "specaugment"}

METADATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "segments_metadata_v1_2.csv"
)

FROZEN_METADATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "segments_metadata_v1_2_train_val_frozen.csv"
)

TOKENIZER_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "xlsr_tokenizer_v1_2"
)

DATASET_CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "xlsr_corpus_v1_2"
)

EXPERIMENT_NAME = (
    "xlsr_300m_tarifit_v1_2_noaug"
    if AUGMENTATION_CONDITION == "noaug"
    else "xlsr_300m_tarifit_v1_2_specaugment"
)

MODEL_DIR = PROJECT_ROOT / "models" / EXPERIMENT_NAME
RESULTS_DIR = PROJECT_ROOT / "results" / EXPERIMENT_NAME

BASE_MODEL_ID = "facebook/wav2vec2-xls-r-300m"
SEED = 42

TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project exists:", PROJECT_ROOT.exists())
print("Metadata exists:", METADATA_PATH.exists())
print("Frozen train/validation metadata exists:", FROZEN_METADATA_PATH.exists())
print("Augmentation condition:", AUGMENTATION_CONDITION)
print("Experiment name:", EXPERIMENT_NAME)
print("Project root:", PROJECT_ROOT)
print("Model output:", MODEL_DIR)
print("Results output:", RESULTS_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project exists: True
Metadata exists: True
Frozen train/validation metadata exists: True
Augmentation condition: noaug
Experiment name: xlsr_300m_tarifit_v1_2_noaug
Project root: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm
Model output: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/xlsr_300m_tarifit_v1_2_noaug
Results output: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/xlsr_300m_tarifit_v1_2_noaug


In [ ]:
# Cell 3 — Record software and GPU environment

import sys
import random
import hashlib

import torch
import transformers
import datasets
import jiwer
import soundfile
import numpy as np

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("JiWER:", jiwer.__version__ if hasattr(jiwer, "__version__") else "unknown")
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    properties = torch.cuda.get_device_properties(0)
    print("GPU memory (GB):", round(properties.total_memory / 1024**3, 2))


Python: 3.13.15
PyTorch: 2.11.0+cu128
Transformers: 4.57.1
Datasets: 4.4.1
JiWER: unknown
CUDA available: True
GPU: NVIDIA L4
GPU memory (GB): 22.03


In [ ]:
# Cell 4 — Load and verify the frozen V1.2 train/validation partitions

import pandas as pd

if not FROZEN_METADATA_PATH.exists():
    raise FileNotFoundError(
        "The frozen V1.2 train/validation metadata was not found. "
        "Do not train until the same frozen split used by the controlled experiments is available."
    )

frozen_df = pd.read_csv(FROZEN_METADATA_PATH)

required_columns = {
    "segment_id",
    "recording_id",
    "speaker_group_id",
    "dataset_split",
    "duration_seconds",
    "audio_path",
    "transcription",
}

missing_columns = required_columns - set(frozen_df.columns)
assert not missing_columns, f"Missing columns: {missing_columns}"

frozen_df["dataset_split"] = (
    frozen_df["dataset_split"].astype(str).str.lower().str.strip()
)
frozen_df["transcription"] = (
    frozen_df["transcription"].fillna("").astype(str).str.strip()
)

assert frozen_df["dataset_split"].isin(["train", "validation"]).all()
assert frozen_df["transcription"].ne("").all()
assert not frozen_df["segment_id"].duplicated().any()

train_df = frozen_df[frozen_df["dataset_split"] == "train"].copy()
validation_df = frozen_df[
    frozen_df["dataset_split"] == "validation"
].copy()

print("Train segments:", len(train_df))
print("Validation segments:", len(validation_df))
print(
    "Train duration:",
    round(train_df["duration_seconds"].sum() / 3600, 3),
    "hours",
)
print(
    "Validation duration:",
    round(validation_df["duration_seconds"].sum() / 3600, 3),
    "hours",
)
print("Train speakers:", sorted(train_df["speaker_group_id"].unique()))
print(
    "Validation speakers:",
    sorted(validation_df["speaker_group_id"].unique()),
)

full_metadata_df = pd.read_csv(METADATA_PATH)
full_metadata_df["dataset_split"] = (
    full_metadata_df["dataset_split"].astype(str).str.lower().str.strip()
)

train_speakers = set(train_df["speaker_group_id"].dropna())
validation_speakers = set(validation_df["speaker_group_id"].dropna())
test_speakers = set(
    full_metadata_df.loc[
        full_metadata_df["dataset_split"] == "test",
        "speaker_group_id",
    ].dropna()
)

assert not train_speakers & validation_speakers
assert not train_speakers & test_speakers
assert not validation_speakers & test_speakers

missing_audio = [
    str(PROJECT_ROOT / path)
    for path in frozen_df["audio_path"]
    if not (PROJECT_ROOT / path).exists()
]

print("Missing audio files:", len(missing_audio))
assert not missing_audio, (
    "Some frozen audio files are missing. First missing file: "
    + (missing_audio[0] if missing_audio else "")
)

metadata_sha256 = hashlib.sha256(
    FROZEN_METADATA_PATH.read_bytes()
).hexdigest()

print("Frozen metadata SHA-256:", metadata_sha256)
print("Speaker-independent partition verification completed.")


Train segments: 1754
Validation segments: 129
Train duration: 5.223 hours
Validation duration: 0.298 hours
Train speakers: ['SPK001', 'SPK002', 'SPK009']
Validation speakers: ['SPK007', 'SPK010']
Missing audio files: 0
Frozen metadata SHA-256: 4911f46e0a8c3656089677b8899d8296b9e1528d8aa3633a64728eb645f28fab
Speaker-independent partition verification completed.


In [ ]:
# Cell 5 — Verify the V1.2 orthographic character inventory

from collections import Counter
import unicodedata

FINAL_LETTERS = (
    list("abcdefghijklmnpqrstuvwxyz")
    + ["ɛ", "ɣ", "ʷ", "ḍ", "ḥ", "ṭ"]
)

ALLOWED_CHARACTERS = set(FINAL_LETTERS) | {" "}

all_text = " ".join(frozen_df["transcription"].tolist())
character_counts = Counter(all_text)

unexpected_characters = {
    character: count
    for character, count in character_counts.items()
    if character not in ALLOWED_CHARACTERS
}

print("Expected letters:", " ".join(FINAL_LETTERS))
print("Number of letters:", len(FINAL_LETTERS))
print("Unexpected characters:", unexpected_characters)

for text in frozen_df["transcription"]:
    assert text == unicodedata.normalize("NFC", text)

assert not unexpected_characters, (
    "The V1.2 transcripts contain characters outside the declared vocabulary."
)

print("V1.2 character inventory verified.")


Expected letters: a b c d e f g h i j k l m n p q r s t u v w x y z ɛ ɣ ʷ ḍ ḥ ṭ
Number of letters: 31
Unexpected characters: {}
V1.2 character inventory verified.


In [ ]:
# Cell 6 — Build and save the XLS-R V1.2 CTC tokenizer

import json
from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
)

vocab_tokens = ["|"] + FINAL_LETTERS + ["[UNK]", "[PAD]"]
vocab_dict = {
    token: index
    for index, token in enumerate(vocab_tokens)
}

VOCAB_PATH = TOKENIZER_DIR / "vocab.json"

with open(VOCAB_PATH, "w", encoding="utf-8") as file:
    json.dump(vocab_dict, file, ensure_ascii=False, indent=2)

tokenizer = Wav2Vec2CTCTokenizer(
    str(VOCAB_PATH),
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
    bos_token=None,
    eos_token=None,
    do_lower_case=False,
)

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True,
)

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)

processor.save_pretrained(TOKENIZER_DIR)

print("Vocabulary path:", VOCAB_PATH)
print("Tokenizer size:", len(tokenizer))
print("CTC blank/padding id:", tokenizer.pad_token_id)
print("Unknown-token id:", tokenizer.unk_token_id)
print("Vocabulary:", tokenizer.get_vocab())


Vocabulary path: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/processed/xlsr_tokenizer_v1_2/vocab.json
Tokenizer size: 34
CTC blank/padding id: 33
Unknown-token id: 32
Vocabulary: {'|': 0, 'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'p': 15, 'q': 16, 'r': 17, 's': 18, 't': 19, 'u': 20, 'v': 21, 'w': 22, 'x': 23, 'y': 24, 'z': 25, 'ɛ': 26, 'ɣ': 27, 'ʷ': 28, 'ḍ': 29, 'ḥ': 30, 'ṭ': 31, '[UNK]': 32, '[PAD]': 33}


In [ ]:
# Cell 7 — Verify that every V1.2 transcript is representable

unknown_segments = []

for row in frozen_df.itertuples(index=False):
    label_ids = tokenizer(row.transcription).input_ids
    if tokenizer.unk_token_id in label_ids:
        unknown_segments.append(
            (row.segment_id, row.transcription)
        )

print("Segments containing [UNK]:", len(unknown_segments))

if unknown_segments:
    for item in unknown_segments[:20]:
        print(item)

assert not unknown_segments, (
    "Some V1.2 references cannot be represented by the tokenizer."
)

print("All V1.2 references are representable.")


Segments containing [UNK]: 0
All V1.2 references are representable.


In [ ]:
# Cell 8 — Build or reload the cached V1.2 XLS-R dataset

import json
import soundfile as sf
from datasets import Dataset, DatasetDict, load_from_disk


def prepare_split(frame):
    work = frame.copy()
    work["absolute_audio_path"] = work["audio_path"].apply(
        lambda path: str(PROJECT_ROOT / path)
    )

    return Dataset.from_pandas(
        work[
            [
                "segment_id",
                "recording_id",
                "speaker_group_id",
                "duration_seconds",
                "absolute_audio_path",
                "transcription",
            ]
        ],
        preserve_index=False,
    )


def prepare_example(example):
    audio, sampling_rate = sf.read(
        example["absolute_audio_path"],
        dtype="float32",
        always_2d=False,
    )

    if sampling_rate != 16000:
        raise ValueError(
            f'{example["segment_id"]}: expected 16000 Hz, '
            f"found {sampling_rate} Hz"
        )

    if getattr(audio, "ndim", 1) != 1:
        raise ValueError(
            f'{example["segment_id"]}: audio is not mono'
        )

    model_inputs = processor(
        audio,
        sampling_rate=16000,
    )

    labels = tokenizer(
        example["transcription"]
    ).input_ids

    return {
        "input_values": model_inputs.input_values[0],
        "input_length": len(model_inputs.input_values[0]),
        "labels": labels,
    }


CACHE_MANIFEST_PATH = DATASET_CACHE_DIR / "cache_manifest.json"

cache_is_valid = False

if DATASET_CACHE_DIR.exists() and CACHE_MANIFEST_PATH.exists():
    with open(CACHE_MANIFEST_PATH, "r", encoding="utf-8") as file:
        cache_manifest = json.load(file)

    cache_is_valid = (
        cache_manifest.get("metadata_sha256") == metadata_sha256
        and cache_manifest.get("tokenizer_size") == len(tokenizer)
        and cache_manifest.get("base_model") == BASE_MODEL_ID
    )

if DATASET_CACHE_DIR.exists() and not cache_is_valid:
    raise RuntimeError(
        "An XLS-R V1.2 cache exists but does not match the frozen metadata, "
        "tokenizer, or base model. Inspect it before choosing a new cache path."
    )

if cache_is_valid:
    xlsr_dataset = load_from_disk(str(DATASET_CACHE_DIR))
    print("Reused validated V1.2 XLS-R dataset cache.")
else:
    raw_dataset = DatasetDict(
        {
            "train": prepare_split(train_df),
            "validation": prepare_split(validation_df),
        }
    )

    xlsr_dataset = raw_dataset.map(
        prepare_example,
        remove_columns=[
            "absolute_audio_path",
            "transcription",
            "recording_id",
            "speaker_group_id",
            "duration_seconds",
        ],
        num_proc=1,
        desc="Preparing V1.2 16 kHz audio and CTC labels",
    )

    DATASET_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    xlsr_dataset.save_to_disk(str(DATASET_CACHE_DIR))

    with open(CACHE_MANIFEST_PATH, "w", encoding="utf-8") as file:
        json.dump(
            {
                "metadata_sha256": metadata_sha256,
                "tokenizer_size": len(tokenizer),
                "base_model": BASE_MODEL_ID,
            },
            file,
            ensure_ascii=False,
            indent=2,
        )

    print("Built and saved V1.2 XLS-R dataset cache.")

train_ds = xlsr_dataset["train"]
validation_ds = xlsr_dataset["validation"]

print("Train examples:", len(train_ds))
print("Validation examples:", len(validation_ds))
print("Dataset columns:", train_ds.column_names)


Preparing V1.2 16 kHz audio and CTC labels (num_proc=1):   0%|          | 0/1754 [00:00<?, ? examples/s]

Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
Exception ignored in: <_io.BytesIO object at 0x788b14dde9d0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/tblib/pickling_support.py", line 13, in unpickle_traceback
    def unpickle_traceback(tb_frame, tb_lineno, tb_next):
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x788b14ddddf0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/tblib/pickling_support.py", line 13, in unpickle_traceback
    def unpickle_traceback(tb_frame, tb_lineno, tb_next):
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x788d81748a40>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/tblib/pickling_support.py", line 13, in unpickle_traceback
    def unpickle_traceback(tb_frame, tb_lineno, tb_next):
BufferErr

Preparing V1.2 16 kHz audio and CTC labels (num_proc=1):   0%|          | 0/129 [00:00<?, ? examples/s]

Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
Exception ignored in: <_io.BytesIO object at 0x788d817491c0>
Traceback (most recent call last):
  File "/usr/lib/python3.13/pickle.py", line 199, in __init__
    def __init__(self, file_write):
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x788b14d44180>
Traceback (most recent call last):
  File "/usr/lib/python3.13/pickle.py", line 199, in __init__
    def __init__(self, file_write):
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x788b14d47ce0>
Traceback (most recent call last):
  File "/usr/lib/python3.13/pickle.py", line 199, in __init__
    def __init__(self, file_write):
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x788b14d45a80>
Traceback (most recent call last):
  File "/usr/lib/python3.13/pickle.py", line 199,

Saving the dataset (0/3 shards):   0%|          | 0/1754 [00:00<?, ? examples/s]

Exception ignored in: <_io.BytesIO object at 0x788d8174af70>
Traceback (most recent call last):
  File "/usr/lib/python3.13/dataclasses.py", line 1372, in fields
    return tuple(f for f in fields.values() if f._field_type is _FIELD)
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x788d8174b100>
Traceback (most recent call last):
  File "/usr/lib/python3.13/dataclasses.py", line 1372, in fields
    return tuple(f for f in fields.values() if f._field_type is _FIELD)
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x788c14a3aed0>
Traceback (most recent call last):
  File "/usr/lib/python3.13/dataclasses.py", line 1372, in fields
    return tuple(f for f in fields.values() if f._field_type is _FIELD)
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x788d8179fb50>
Traceback (most recent call last):
  File 

Saving the dataset (0/1 shards):   0%|          | 0/129 [00:00<?, ? examples/s]

Built and saved V1.2 XLS-R dataset cache.
Train examples: 1754
Validation examples: 129
Dataset columns: ['segment_id', 'input_values', 'input_length', 'labels']


In [ ]:
# Cell 9 — Summarize the processed V1.2 audio durations

def duration_summary(dataset_split):
    seconds = np.asarray(
        dataset_split["input_length"],
        dtype=np.float64,
    ) / 16000.0

    return {
        "segments": len(seconds),
        "hours": float(seconds.sum() / 3600),
        "minimum_seconds": float(seconds.min()),
        "maximum_seconds": float(seconds.max()),
        "mean_seconds": float(seconds.mean()),
    }


print("Train:", duration_summary(train_ds))
print("Validation:", duration_summary(validation_ds))


Train: {'segments': 1754, 'hours': 5.222733593749999, 'minimum_seconds': 0.848, 'maximum_seconds': 19.984, 'mean_seconds': 10.71940760404789}
Validation: {'segments': 129, 'hours': 0.2983577777777778, 'minimum_seconds': 0.944, 'maximum_seconds': 26.0, 'mean_seconds': 8.326263565891473}


In [ ]:
# Cell 10 — Load a fresh XLS-R 300M CTC model for V1.2

from transformers import Wav2Vec2ForCTC

if AUGMENTATION_CONDITION == "noaug":
    APPLY_SPEC_AUGMENT = False
    MASK_TIME_PROB = 0.0
    MASK_TIME_LENGTH = 5
else:
    APPLY_SPEC_AUGMENT = True
    MASK_TIME_PROB = 0.05
    MASK_TIME_LENGTH = 5

model = Wav2Vec2ForCTC.from_pretrained(
    BASE_MODEL_ID,
    vocab_size=len(tokenizer),
    pad_token_id=tokenizer.pad_token_id,
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,
    attention_dropout=0.0,
    hidden_dropout=0.0,
    feat_proj_dropout=0.0,
    apply_spec_augment=APPLY_SPEC_AUGMENT,
    mask_time_prob=MASK_TIME_PROB,
    mask_time_length=MASK_TIME_LENGTH,
    mask_feature_prob=0.0,
    layerdrop=0.0,
    ignore_mismatched_sizes=True,
)

# Keep the convolutional feature encoder frozen, as in the previous X2 run.
model.freeze_feature_encoder()

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)
trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Base model:", BASE_MODEL_ID)
print("Total parameters:", f"{total_parameters:,}")
print("Trainable parameters:", f"{trainable_parameters:,}")
print(
    "Trainable percentage:",
    f"{100 * trainable_parameters / total_parameters:.4f}%",
)
print("External waveform augmentation: none")
print("Internal SpecAugment enabled:", model.config.apply_spec_augment)
print("Internal time-mask probability:", model.config.mask_time_prob)
print("Internal time-mask length:", model.config.mask_time_length)


Exception ignored in: <_io.BytesIO object at 0x788d8177cfe0>
Traceback (most recent call last):
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x788d8177ffb0>
Traceback (most recent call last):
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x788b14d45d50>
Traceback (most recent call last):
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x788d8177f4c0>
Traceback (most recent call last):
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO objec

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-xls-r-300m and are newly initialized: ['lm_head.bias', 'lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Base model: facebook/wav2vec2-xls-r-300m
Total parameters: 315,472,546
Trainable parameters: 311,262,370
Trainable percentage: 98.6654%
External waveform augmentation: none
Internal SpecAugment enabled: False
Internal time-mask probability: 0.0
Internal time-mask length: 5


In [ ]:
# Cell 11 — Check CTC feasibility of every V1.2 example

def minimum_ctc_frames(labels):
    repeated_adjacent_labels = sum(
        labels[index] == labels[index - 1]
        for index in range(1, len(labels))
    )
    return len(labels) + repeated_adjacent_labels


def find_ctc_infeasible(dataset_split):
    invalid = []

    for index, example in enumerate(dataset_split):
        input_samples = int(example["input_length"])
        output_frames = int(
            model._get_feat_extract_output_lengths(
                torch.tensor(input_samples)
            ).item()
        )

        labels = example["labels"]
        required_frames = minimum_ctc_frames(labels)

        if output_frames < required_frames:
            invalid.append(
                {
                    "index": index,
                    "segment_id": example.get("segment_id", index),
                    "audio_seconds": input_samples / 16000.0,
                    "output_frames": output_frames,
                    "label_length": len(labels),
                    "minimum_ctc_frames": required_frames,
                }
            )

    return invalid


bad_train = find_ctc_infeasible(train_ds)
bad_validation = find_ctc_infeasible(validation_ds)

print("CTC-infeasible training examples:", len(bad_train))
print("CTC-infeasible validation examples:", len(bad_validation))

if bad_train:
    display(pd.DataFrame(bad_train))
if bad_validation:
    display(pd.DataFrame(bad_validation))

assert not bad_train, (
    "Training contains CTC-infeasible examples. Inspect before training."
)
assert not bad_validation, (
    "Validation contains CTC-infeasible examples. Inspect before training."
)

print("All V1.2 examples are CTC-feasible.")


model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

CTC-infeasible training examples: 0
CTC-infeasible validation examples: 0
All V1.2 examples are CTC-feasible.


In [ ]:
# Cell 12 — Define dynamic CTC padding for audio and labels

from dataclasses import dataclass
from typing import Dict, List, Union


@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(
        self,
        features: List[Dict[str, Union[List[int], torch.Tensor]]],
    ) -> Dict[str, torch.Tensor]:
        input_features = [
            {"input_values": feature["input_values"]}
            for feature in features
        ]

        label_features = [
            {"input_ids": feature["labels"]}
            for feature in features
        ]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["attention_mask"].ne(1),
            -100,
        )

        batch["labels"] = labels
        return batch


data_collator = DataCollatorCTCWithPadding(
    processor=processor,
    padding=True,
)

print("CTC data collator ready.")


CTC data collator ready.


In [ ]:
# Cell 13 — Define validation WER and CER metrics

from jiwer import wer, cer


def compute_metrics(prediction):
    prediction_ids = np.argmax(
        prediction.predictions,
        axis=-1,
    )

    label_ids = prediction.label_ids.copy()
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    prediction_text = tokenizer.batch_decode(prediction_ids)
    reference_text = tokenizer.batch_decode(
        label_ids,
        group_tokens=False,
    )

    return {
        "wer": wer(reference_text, prediction_text) * 100,
        "cer": cer(reference_text, prediction_text) * 100,
    }


print("WER/CER metrics ready.")


WER/CER metrics ready.


In [ ]:
# Cell 14 — Save the XLS-R V1.2 experiment configuration

experiment_configuration = {
    "experiment": EXPERIMENT_NAME,
    "base_model": BASE_MODEL_ID,
    "augmentation_condition": AUGMENTATION_CONDITION,
    "metadata_path": str(FROZEN_METADATA_PATH),
    "metadata_sha256": metadata_sha256,
    "tokenizer_path": str(TOKENIZER_DIR),
    "tokenizer_size": len(tokenizer),
    "final_letters": FINAL_LETTERS,
    "seed": SEED,
    "external_waveform_augmentation": False,
    "internal_spec_augment": APPLY_SPEC_AUGMENT,
    "internal_mask_time_prob": MASK_TIME_PROB,
    "internal_mask_time_length": MASK_TIME_LENGTH,
    "feature_encoder_frozen": True,
    "planned_epochs": 8,
    "per_device_train_batch_size": 2,
    "per_device_eval_batch_size": 2,
    "gradient_accumulation_steps": 8,
    "effective_train_batch_size": 16,
    "learning_rate": 3e-5,
    "weight_decay": 0.01,
    "warmup_steps": 100,
    "fp16": bool(torch.cuda.is_available()),
    "gradient_checkpointing": True,
    "checkpoint_selection_metric": "validation CER",
}

CONFIGURATION_PATH = RESULTS_DIR / "experiment_configuration.json"

with open(CONFIGURATION_PATH, "w", encoding="utf-8") as file:
    json.dump(
        experiment_configuration,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("Saved configuration:", CONFIGURATION_PATH)


Saved configuration: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/xlsr_300m_tarifit_v1_2_noaug/experiment_configuration.json


In [ ]:
# Cell 15 — Configure the XLS-R V1.2 Trainer

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=str(MODEL_DIR),
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    group_by_length=True,
    length_column_name="input_length",
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_steps=100,
    lr_scheduler_type="linear",
    num_train_epochs=8,
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,
    logging_strategy="steps",
    logging_steps=25,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
    dataloader_num_workers=2,
    remove_unused_columns=True,
)

print(training_args)


TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=2,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [ ]:
# Cell 16 — Create the XLS-R V1.2 Trainer with early stopping

from transformers import EarlyStoppingCallback, Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    train_dataset=train_ds,
    eval_dataset=validation_ds,
    processing_class=processor,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2,
            early_stopping_threshold=0.0,
        )
    ],
)

print("Trainer ready.")
print("Train examples:", len(train_ds))
print("Validation examples:", len(validation_ds))
print("Planned epochs:", training_args.num_train_epochs)


Trainer ready.
Train examples: 1754
Validation examples: 129
Planned epochs: 8


In [ ]:
# Cell 17 — Start XLS-R V1.2 fine-tuning

train_result = trainer.train()

print("Training completed.")
print(train_result)
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation CER:", trainer.state.best_metric)


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.
Exception ignored in: <_io.BytesIO object at 0x788d8177ccc0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/accelerate/utils/memory.py", line 46, in clear_device_cache
    gc.collect()
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x788b14ddf150>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/accelerate/utils/memory.py", line 46, in clear_device_cache
    gc.collect()
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x788d81748b80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/accelerate/utils

Epoch,Training Loss,Validation Loss,Wer,Cer
1,3.917300,3.875505,100.000000,100.000000
2,3.060800,3.309290,100.000000,100.000000
3,2.943800,3.258047,100.000000,100.000000


/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Training completed.
TrainOutput(global_step=330, training_loss=4.6547625310493235, metrics={'train_runtime': 549.7755, 'train_samples_per_second': 25.523, 'train_steps_per_second': 1.601, 'total_flos': 1.7124415098444628e+18, 'train_loss': 4.6547625310493235, 'epoch': 3.0})
Best checkpoint: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/xlsr_300m_tarifit_v1_2_noaug/checkpoint-110
Best validation CER: 100.0


In [ ]:
# Cell 18 — Evaluate and save the best XLS-R V1.2 checkpoint

best_validation_metrics = trainer.evaluate(
    eval_dataset=validation_ds,
    metric_key_prefix="validation",
)

print("Best-checkpoint validation metrics:")
for key, value in best_validation_metrics.items():
    print(f"{key}: {value}")

BEST_MODEL_DIR = MODEL_DIR / "best_model"
trainer.save_model(str(BEST_MODEL_DIR))
processor.save_pretrained(BEST_MODEL_DIR)

print("Saved best model:", BEST_MODEL_DIR)


In [ ]:
# Cell 19 — Save V1.2 validation predictions for error analysis

prediction_output = trainer.predict(
    validation_ds,
    metric_key_prefix="validation_prediction",
)

prediction_ids = np.argmax(
    prediction_output.predictions,
    axis=-1,
)

prediction_texts = tokenizer.batch_decode(prediction_ids)
reference_texts = [
    tokenizer.decode(
        example["labels"],
        group_tokens=False,
    )
    for example in validation_ds
]

validation_predictions = pd.DataFrame(
    {
        "segment_id": validation_ds["segment_id"],
        "reference": reference_texts,
        "prediction": prediction_texts,
    }
)

PREDICTIONS_PATH = RESULTS_DIR / "validation_predictions.csv"
validation_predictions.to_csv(
    PREDICTIONS_PATH,
    index=False,
    encoding="utf-8",
)

display(validation_predictions.head(20))
print("Saved predictions:", PREDICTIONS_PATH)


In [ ]:
# Cell 20 — Save the training history and final experiment summary

history_df = pd.DataFrame(trainer.state.log_history)
HISTORY_PATH = RESULTS_DIR / "training_history.csv"
history_df.to_csv(HISTORY_PATH, index=False)

summary = {
    **experiment_configuration,
    "train_segments": len(train_ds),
    "validation_segments": len(validation_ds),
    "train_duration": duration_summary(train_ds),
    "validation_duration": duration_summary(validation_ds),
    "total_parameters": int(total_parameters),
    "trainable_parameters": int(trainable_parameters),
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "best_validation_metric": (
        float(trainer.state.best_metric)
        if trainer.state.best_metric is not None
        else None
    ),
    "final_validation_metrics": {
        key: float(value)
        if isinstance(value, (int, float, np.floating))
        else value
        for key, value in best_validation_metrics.items()
    },
    "results": {
        "training_history": str(HISTORY_PATH),
        "validation_predictions": str(PREDICTIONS_PATH),
        "best_model": str(BEST_MODEL_DIR),
    },
}

SUMMARY_PATH = RESULTS_DIR / "experiment_summary.json"
with open(SUMMARY_PATH, "w", encoding="utf-8") as file:
    json.dump(summary, file, ensure_ascii=False, indent=2)

print("Saved training history:", HISTORY_PATH)
print("Saved experiment summary:", SUMMARY_PATH)
